<a href="https://colab.research.google.com/github/Ashu251023/DETraining/blob/Dev/Pyspark_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark findspark

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PysparkPractice").getOrCreate()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DecimalType
from datetime import datetime

spark = SparkSession.builder.appName("EmployeesData").getOrCreate()

schema = StructType([
    StructField("employeeId", IntegerType(), True),
    StructField("employeeName", StringType(), True),
    StructField("employeeSurname", StringType(), True),
    StructField("employeeTitle", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("birthdate", DateType(), True),
    StructField("salary", IntegerType(), True)
])

data = [
    (1,"John", "Doe", "Manager", 35, "New York", "1986-05-10", 75000),
    (2,"Jane", "Doe", "Developer", 32, "London", "1989-02-15", 65000),
    (3,"Jim", "Smith", "Architect", 40, "Paris", "1980-08-20", 85000),
    (4,"Sarah", "Johnson", "Designer", 29, "Berlin", "1992-12-01", 55000),
    (5,"Michael", "Brown", "Product Manager", 38, "Tokyo", "1984-06-06", 75000),
    (6,"Emily", "Davis", "Data Analyst", 31, "Sydney", "1990-09-12", 65000),
    (7,"David", "Wilson", "Salesperson", 33, "Toronto", "1988-07-01", 55000),
    (8,"William", "Johnson", "Support Engineer", 36, "Beijing", "1985-04-01", 65000),
    (9,"Brian", "Anderson", "Marketing Manager", 37, "Shanghai", "1983-05-15", 75000),
    (10,"James", "Lee", "Operations Manager", 39, "Seoul", "1981-03-01", 85000),
    (11,"Emily", "Parker", "HR Manager", 30, "Dubai", "1991-12-25", 75000),
    (12,"Jacob", "Williams", "Accountant", 34, "New Delhi", "1987-06-01", 65000)
]

data = [(r[0], r[1], r[2], r[3], r[4], r[5], datetime.strptime(r[6], "%Y-%m-%d").date(), r[7]) for r in data]

df_employees = spark.createDataFrame(data, schema=schema)
df_employees.show()


+----------+------------+---------------+------------------+---+---------+----------+------+
|employeeId|employeeName|employeeSurname|     employeeTitle|age|     city| birthdate|salary|
+----------+------------+---------------+------------------+---+---------+----------+------+
|         1|        John|            Doe|           Manager| 35| New York|1986-05-10| 75000|
|         2|        Jane|            Doe|         Developer| 32|   London|1989-02-15| 65000|
|         3|         Jim|          Smith|         Architect| 40|    Paris|1980-08-20| 85000|
|         4|       Sarah|        Johnson|          Designer| 29|   Berlin|1992-12-01| 55000|
|         5|     Michael|          Brown|   Product Manager| 38|    Tokyo|1984-06-06| 75000|
|         6|       Emily|          Davis|      Data Analyst| 31|   Sydney|1990-09-12| 65000|
|         7|       David|         Wilson|       Salesperson| 33|  Toronto|1988-07-01| 55000|
|         8|     William|        Johnson|  Support Engineer| 36|  Beij

In [ ]:

# Preprocess and Create DataFrame
data = [(r[0], r[1], r[2], r[3], r[4], r[5], datetime.strptime(r[6], "%Y-%m-%d").date(), r[7]) for r in data]
df = spark.createDataFrame(data, schema=schema)

# 1. Employees older than 35
df.filter(col("age") > 35).show()

# 2. Unique job titles
df.select("employeeTitle").distinct().show()

# 3. Average salary
df.select(avg("salary")).show()

# 4. Employees born after 1985
df.filter(year("birthdate") > 1985).show()

# 5. Count by city
df.groupBy("city").count().show()

# 6. Sort by salary descending
df.orderBy(col("salary").desc()).show()

# 7. Add age group column
df.withColumn("age_group",
    when(col("age") < 30, "<30")
    .when((col("age") >= 30) & (col("age") <= 40), "30-40")
    .otherwise(">40")).show()

# 8. Filter by Manager in title
df.filter(col("employeeTitle").contains("Manager")).show()

# 9. Count employees by job title
df.groupBy("employeeTitle").count().show()

# 10. Top 5 highest paid employees
df.orderBy(col("salary").desc()).limit(5).show()

# 11. Add birth year column
df.withColumn("birth_year", year("birthdate")).show()

# 12. Salary between 60000 and 80000
df.filter((col("salary") >= 60000) & (col("salary") <= 80000)).show()

# 13. Count by city (simulating countries)
df.groupBy("city").count().show()

# 14. Youngest and oldest employees
df.orderBy("age").select("employeeName", "age").show(1)
df.orderBy(col("age").desc()).select("employeeName", "age").show(1)

# 15. Replace "Tokyo" with "Osaka"
df.withColumn("city", when(col("city") == "Tokyo", "Osaka").otherwise(col("city"))).show()

# 16. Create full name
df.withColumn("fullName", concat_ws(" ", col("employeeName"), col("employeeSurname"))).show()

# 17. Total salary of all employees
df.select(_sum("salary").alias("total_salary")).show()

# 18. Duplicate first names
df.groupBy("employeeName").count().filter("count > 1").show()

# 19. Unique salaries
df.groupBy("salary").count().filter("count = 1").show()

# 20. Salary percent of total
total_salary = df.agg(_sum("salary").alias("total")).collect()[0]["total"]
df.withColumn("salary_percent", (col("salary") / total_salary) * 100).show()

# 21. SQL Querying
df.createOrReplaceTempView("employees")
spark.sql("SELECT city, COUNT(*) as num_employees FROM employees GROUP BY city").show()

# 22. Salary above average flag
avg_salary = df.agg(avg("salary").alias("avg_salary")).collect()[0]["avg_salary"]
df.withColumn("above_avg", col("salary") > avg_salary).show()